<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🔬 M2 Lab 1 · 让 Deep Research Crew 更可靠

**一句话**：在 Module 1 的 4-agent 研究 crew 上加三样东西：最终报告的 **guardrail**、存文件的 **after_kickoff hook**、以及 **memory**。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ Lab</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 C1M2_Lab_1_deep_research_p1.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：M1 deep research crew</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 config/agents.yaml · config/tasks.yaml · utils.py</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：作为研究顾问，你要给客户交「查过资料、核过事实、结构完整」的报告。
上一版 crew 能跑，但输出没保障：报告可能漏掉摘要或引用、结果只在屏幕上、每次运行从零开始。
本 lab 补上：**guardrail**（报告必须有 Summary / Insights / Citations 三节）、**hook**（自动存成 `research_report.md`）、**memory**（跨 task、跨运行记住东西）。

需要你填的（都已填好，保留你的答案）：guardrail 的 Citations 检查、给 `write_final_report` 挂 guardrail、Crew 的 `memory` 和 `after_kickoff_callbacks`、你的 `user_query`。

</div>

<img src="./images/lab1-agents-tasks-diagram.png" width="900">

<details><summary>竖版示意图（你的截图）</summary>

<img src="./Screenshot 2026-10-11 at 22.37.51_enhanced.png" width="420">

</details>

| agent | 负责的 task | 工具 | 输出交给谁 |
|---|---|---|---|
| Research Planner | `create_research_plan`：拆成研究主题 + 关键问题 + 成功标准 | 无 | Internet Researcher |
| Internet Researcher | `gather_research_data`：搜索 + 抓取网页，带引用 | `EXASearchTool` `ScrapeWebsiteTool` | Fact Checker |
| Fact Checker | `verify_information_quality`：交叉验证、标出可疑信息与缺口 | 同上 | Report Writer |
| Report Writer | `write_final_report`：写最终报告（摘要 / 发现 / 证据 / 引用 / 建议） | 无（Lab 2 加画图工具） | `after_kickoff` hook 存文件 |

默认 `process=sequential`：task 按列表顺序执行，**前面所有 task 的输出自动作为后面 task 的 context**，所以 tasks.yaml 里不用写 `context`。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 三个新零件：Guardrail · Hook · Memory

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #f87171; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🛡️ Task Guardrail</div><div style="font-size:0.92em; line-height:1.6;">挂在 <b>某个 task</b> 上的检查函数。<br>task 产出 → 先过 guardrail → 通过才交给下一个 task。<br>不通过 → 把错误文字当反馈让 agent <b>重做</b>（默认最多 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">guardrail_max_retries=3</code> 次，超了抛异常）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🪝 after_kickoff hook</div><div style="font-size:0.92em; line-height:1.6;">挂在 <b>整个 Crew</b> 上的回调：所有 task 跑完后自动调用，参数是 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">CrewOutput</code>。<br>本 lab 用它把最终报告存成 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">.md</code> 文件。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧠 Memory</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">Crew(memory=True)</code>：每个 task 开始前按 task 描述 <b>recall</b> 相关记忆拼进 prompt，结束后把结果提炼成记忆 <b>存盘</b>。<br>需要 embedding（默认 OpenAI）。</div></div></div></div>

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">guardrail 通过</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #a78bfa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">task 输出 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">TaskOutput</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">guardrail(output)</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">(True, 值)</code> → 值成为该 task 的最终输出</span></div>

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">guardrail 不通过</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">guardrail(output)</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fb923c; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">(False, &quot;错误说明&quot;)</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">错误说明作为反馈 → agent 重做</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">再检查（最多 3 次重试）</span></div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📄 配置文件：`config/agents.yaml` 与 `config/tasks.yaml`

</div>

agent 的 `role / goal / backstory` 和 task 的 `description / expected_output` 都写在 YAML 里，Python 只负责「装配」：

| 文件 | 顶层 key | 每个 key 下的字段 | Python 里怎么用 |
|---|---|---|---|
| `agents.yaml` | `research_planner` `internet_researcher` `fact_checker` `report_writer` | `role` `goal` `backstory` | `Agent(config=agent_config["research_planner"], ...)` |
| `tasks.yaml` | `create_research_plan` `gather_research_data` `verify_information_quality` `write_final_report` | `description` `expected_output` | `Task(config=task_config["..."], agent=...)` |

- `config=` 字典里的字段等价于直接写 `Agent(role=..., goal=...)`；工具、`verbose`、agent 绑定等 **运行时对象** YAML 表达不了，所以在 Python 里补。
- `{user_query}` 是占位符：`kickoff(inputs={"user_query": ...})` 时被替换进 **所有** agent / task 文本。
- YAML 语法：`'...'` 单引号字符串和 `>` 折叠块都可以跨行，换行会被折成空格。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 1 步：导入 + 环境变量

</div>

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔑 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">utils.get_*_api_key()</code></div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">load_dotenv(find_dotenv())</code>：从当前目录往上找 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">.env</code>（找到仓库根的那份），再读环境变量。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🤖 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">os.environ[&quot;MODEL&quot;]</code></div><div style="font-size:0.92em; line-height:1.6;">没给 agent 传 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">llm=</code> 时，CrewAI 用 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">MODEL</code> 环境变量当默认模型（不设则用 crewai 内置默认，1.15.22 是 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">gpt-4.1-mini</code>）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧪 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">CREWAI_TESTING=true</code></div><div style="font-size:0.92em; line-height:1.6;">让 crewai 认为处于测试环境：<b>跳过首次运行时「是否开启 tracing」的交互确认</b>，notebook 不会卡住等输入。</div></div></div></div>

In [1]:
from crewai import Agent, Task, Crew
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
os.environ["CREWAI_TESTING"] = "true"   # 跳过首次运行的 tracing 交互确认
from utils import get_openai_api_key, get_exa_api_key
from IPython.display import Markdown
import yaml

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"   # 所有没指定 llm 的 agent 默认用它
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the EXA API key
os.environ["EXA_API_KEY"] = get_exa_api_key()   # EXASearchTool 从这个环境变量读 key

先看一眼两个 YAML 配置（只是读文件打印，帮助理解上面的表）：

In [2]:
from pathlib import Path
print(Path("config/agents.yaml").read_text())   # 4 个 agent 的 role / goal / backstory
print(Path("config/tasks.yaml").read_text())    # 4 个 task 的 description / expected_output

---
research_planner:
  role: Research Planner
  goal: Analyze "{user_query}" and break it down into specific research topics and
    key questions that need investigation
  backstory: You are a research strategist who excels at breaking down complex questions
    into manageable research components. You identify what needs to be researched
    and create clear research objectives.
internet_researcher:
  role: Internet Researcher
  goal: 'Research assigned topics using search and web scraping tools to gather comprehensive,
    accurate information from multiple credible sources. Ensure all information is properly cited and verified.'
  backstory: You are a skilled researcher with experience in online investigation
    and data collection. You know how to find reliable sources, extract relevant information,
    and always verify facts across multiple sources to avoid misinformation or hallucination.
    You never invent facts and always trace information to its origin.
fact_checker:
  r

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 2 步：工具 + 4 个 agent

</div>

两个课程视频里没讲、lab 里加上的参数（都是为了跑得快、别撞限流）：
- `max_rpm=150`：该 agent 每分钟最多 150 次 LLM 请求，超了自动等待；
- `max_iter=15`：思考↔调工具 最多 15 轮（默认 25），到上限必须给出「当前最好答案」。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异**：`EXASearchTool` 在 crewai_tools（随 crewai 1.15.22）里已标记 deprecated，会打印 `DeprecationWarning ... use ExaSearchTool instead`，但仍可用（它的 `name` 已经是 `ExaSearchTool`）。
`base_url=os.getenv("EXA_BASE_URL")` 是课程平台的代理地址；本地没设这个变量 → `None` → 直连 Exa 官方 API。

</div>

In [3]:
# create the tool instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))    # 语义搜索：参数 search_query，返回网页摘要+链接
scrape_website_tool = ScrapeWebsiteTool()   # 抓网页正文：参数 website_url

# load the configuration file for the agents
with open('config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)   # → {'research_planner': {...}, ...}

# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],   # role/goal/backstory 来自 YAML
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
internet_researcher = Agent(
        config=agent_config['internet_researcher'],
        tools=[exa_search_tool, scrape_website_tool],   # 只有 researcher 和 fact checker 能上网
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
fact_checker = Agent(
        config=agent_config['fact_checker'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )

/var/folders/xx/ypnl5f2n0y7b48w_pgxyhqt80000gn/T/ipykernel_35921/329398689.py:2: DeprecationWarning: EXASearchTool is deprecated and will be removed in a future release; use ExaSearchTool instead.
  exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))    # 语义搜索：参数 search_query，返回网页摘要+链接


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 3 步：写 guardrail —— 报告必须有三节

</div>

规则（来自文档）：guardrail 函数 **只能有一个参数**（被检查的 `TaskOutput`），返回 `(bool, Any)`：
- 通过 → `(True, 要传下去的结果)`（可以顺便改写结果）；
- 失败 → `(False, "错误说明")`，这段文字会作为反馈让 agent 重写。

「是标题」用正则判断：`#+.*summary` = 一个或多个 `#` 后面同一行出现 summary。整段先 `.lower()`，所以大小写都能匹配。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

📝 **你填的部分**：Citations / References 检查（`re.search(r'#+.*citations|#+.*references', ...)`）——正确，与 Solution 一致。

</div>

In [4]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw    # 测试时传 str；crew 运行时传 TaskOutput，取 .raw 原文
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()   # 统一小写，Summary/SUMMARY 都能匹配

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):   # 标题行里含 summary（如 '## Executive Summary'）
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    ### START CODE HERE ###

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower):
        return (False, 
                "The report must include a Citations section with a header like '## Citations'"
                )

    ### END CODE HERE ###
    return (True, output)   # 通过：原样把报告传下去

用两份假报告测一下：第一份三节齐全 → `(True, ...)`；第二份只有摘要 → 在 Insights 那一步就失败。

In [5]:
test_report_pass = """
# Report title

## Executive Summary
This is a summary.

## Insights
These are the insights.

## Citations
1. Citation 1
2. Citation 2
"""

write_report_guardrail(test_report_pass)

(True,
 '\n# Report title\n\n## Executive Summary\nThis is a summary.\n\n## Insights\nThese are the insights.\n\n## Citations\n1. Citation 1\n2. Citation 2\n')

In [6]:
test_report_fail = """
# Report title

## Executive Summary
This is a summary.
"""

write_report_guardrail(test_report_fail)

(False,
 "The report must include an Insights section with a header like '## Insights'")

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**guardrail 有局限**：`re.search` 搜的是整篇报告，`.*` 不跨行，但只要 **任意一行** 里 `#` 后面出现关键词就算过（比如正文里有 `#1 recommendations`）。足够挡住「整节漏掉」，挡不住「标题写得怪」。
另一种写法：`guardrails=["报告必须包含...的自然语言描述"]` —— 传字符串时 crewai 会自动建一个 `LLMGuardrail`，用 agent 的 LLM 来判定。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 4 步：从 YAML 建 4 个 task，并挂上 guardrail

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

📝 **你填的部分**：每个 task 绑定对应 agent；`write_final_report` 加 `guardrails=[write_report_guardrail]`——正确，与 Solution 一致。

</div>

In [7]:
# load the configuration file for the tasks
with open('config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

### START CODE HERE ###

# create the tasks using the configuration
create_research_plan = Task( 
    config=task_config['create_research_plan'],
    agent=research_planner   # 谁来做这个 task
)

gather_research_data = Task( 
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task( 
    config=task_config['verify_information_quality'],
    agent=fact_checker,
)

write_final_report = Task( 
    config=task_config['write_final_report'],
    agent=report_writer,
    guardrails=[write_report_guardrail], # add the custom guardrail   # 列表：可挂多个，按顺序依次检查
)

### END CODE HERE ###

`guardrails=` 是列表（也可用单数 `guardrail=` 传一个）；失败重试次数由 `guardrail_max_retries`（默认 3）控制，用完仍失败会抛 `Exception: Task failed ... validation after 3 retries`。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 5 步：after_kickoff hook —— 把报告存成文件

</div>

`after_kickoff_callbacks` 里的函数在 **所有 task 结束后** 被调用，参数 `result` 是 `CrewOutput`：
`result.tasks_output` = 每个 task 的 `TaskOutput` 列表，`[-1].raw` 就是最终报告的 Markdown 原文。
（对应还有 `before_kickoff_callbacks`：kickoff 前调用，可以修改 `inputs`。）

In [8]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw   # 最后一个 task（写报告）的原文
        else:
            report_content = str(result)
        
        filename = f"research_report.md"   # 相对路径 → 存在 notebook 所在文件夹
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

    return result   # 🆕 必须把 result 还回去：hook 的返回值会替换 kickoff 的返回值（原代码没有这行 → result 变成 None）

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 修正课程代码：hook 末尾加了 `return result`**
crewai 源码（1.15.22 `crew.py`）里是 `for cb in self.after_kickoff_callbacks: result = cb(result)` ——每个 after-kickoff 回调的 **返回值会替换** crew 的最终结果（字段说明："It may be used to adjust the output of the crew"）。
课程的 `save_file_hook` 只存文件、不返回 → `kickoff` 返回 `None`，后面再用 `result.raw` 就会 `AttributeError: 'NoneType' object has no attribute ...`（课程 notebook 恰好没再用 `result`，所以没暴露）。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 6 步：组装 Crew（memory + hook）

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

📝 **你填的部分**：`memory=True` 与 `after_kickoff_callbacks=[save_file_hook]`——正确。（`# Create the urban planning crew` 是课程从别的 lab 复制来的旧注释，和本 crew 无关。）

</div>

In [9]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[
        research_planner,
        internet_researcher,
        fact_checker,
        report_writer
    ],
    # include all the tasks in the order to be executed
    tasks=[   # 默认 Process.sequential：按此顺序执行，前序输出自动成为 context
        create_research_plan,
        gather_research_data,
        verify_information_quality,
        write_final_report,
    ],
    ### START CODE HERE ###
    # add memory to the crew
    memory=True,
    # add the after kickoff hook
    # ! 给整个 Crew 注册一个"执行完所有 task 之后自动调用的回调函数
    after_kickoff_callbacks=[save_file_hook],
    ### END CODE HERE ###
)

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026-10 版本差异：`memory=True` 背后是什么**（已对照 crewai 1.15.22 源码）
- 旧版课程时代：short-term / long-term / entity 三种记忆（ChromaDB + SQLite）。
- 现在：统一的 `Memory` 类（`crewai/memory/unified_memory.py`）——存储默认 **LanceDB**，embedding 默认 OpenAI `text-embedding-3-large`，存盘时还会用 agent 的 LLM 分析内容、打分类和重要度。
- 存在 `~/Library/Application Support/<notebook 所在文件夹名>/memory`（`crewai_core.paths.db_storage_path()` 用 `Path.cwd().name` 当 app 名；设 `CREWAI_STORAGE_DIR` 可改），**跨运行持久**：同一文件夹、同名 crew（默认 scope `/crew/crew`）再次运行会召回上次的记忆。
- 召回点：每个 task 开始前用 task 描述 `recall(limit=5)`，命中的记忆以 "Relevant memories:" 拼进 prompt。
- 还会给每个 agent **自动加两个工具** `Search memory` / `Save to memory`（`crewai/tools/memory_tools.py`），所以 verbose 日志里会看到 agent 自己调用 memory 工具（"Found memories" / "Saving N items to memory"）。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 7 步：输入 + 运行

</div>

`inputs` 的 key 必须和 YAML 里的占位符同名（`{user_query}`）。想换题目就改这里。

In [10]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = {
    "user_query": "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."   # ✏️ 改这里：你的研究问题
}

### END CODE HERE ###

运行要几分钟：planner → researcher（多次 Exa 搜索 + 抓网页）→ fact checker（再搜一遍核对）→ writer（过 guardrail）→ hook 存文件。`verbose=True` 会打印每个 agent 的思考和工具调用。

In [11]:
# Execute the crew's tasks
result = await deep_research_crew.kickoff_async(inputs=inputs)   # notebook 里必须用 async 版本（见下方笔记）

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Evaluate the top five emerging AI tools for automating competitive        │
│  market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing   │
│  firm." into specific topics and key questions that need investigation. Create a focused research plan.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Found memories:
- (score=0.83) The report evaluates the top five emerging AI tools for automating competitive market analysis: AlphaSense, Similarweb, Crayon, GWI Spark, and Perplexity AI.
  categorie...
Tool search_memory executed with result: Found memories:
- (score=0.75) AlphaSense costs between $10,000 and $40,000 annually and is ideal for corporate strategy for investors needing in-depth market intelligence.
  categories: market analys...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Research Plan for Evaluating Emerging AI Tools for Automating Competitive Market Analysis                  │
│                                                                                                                 │
│  #### 1. Main Research Topics to Investigate                                                                    │
│                                                                                                                 │
│  1. **Identification of Top AI Tools**                                                                          │
│     - Focus on the top five emerging AI tools for automating competitive market analysis.                       │
│                                                                                                                 │
│  2. **Features of AI Tools**                                                                                    │
│     - Investigate the key functionalities and offerings of the identified AI tools.                             │
│                                                                                                                 │
│  3. **Limitations of AI Tools**                                                                                 │
│     - Analyze the drawbacks and potential challenges associated with each tool.                                 │
│                                                                                                                 │
│  4. **Cost Analysis**                                                                                           │
│     - Examine the pricing structures of each tool, including subscription models and any hidden costs.          │
│                                                                                                                 │
│  5. **Ideal Use Cases for Mid-Sized Marketing Firms**                                                           │
│     - Assess the scenarios in which each tool would be optimally used by mid-sized marketing firms.             │
│                                                                                                                 │
│  #### 2. Key Questions for Each Topic                                                                           │
│                                                                                                                 │
│  1. **Identification of Top AI Tools**                                                                          │
│     - What are the top five emerging AI tools currently available for automating competitive market analysis?   │
│     - What criteria were used to select these tools?                                                            │
│                                                                                                                 │
│  2. **Features of AI Tools**                                                                                    │
│     - What unique features do these tools offer for competitive market analysis?                                │
│     - How do they differentiate themselves from traditional market analysis methods?                            │
│                                                                                                                 │
│  3. **Limitations of AI Tools**                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Task: Create a comprehensive report that answers the original query "Evaluate the top five emerging AI tools   │
│  for automating competitive market analysis, including their features, limitations, costs, and ideal use cases  │
│  for a mid-sized marketing firm." using all verified research data. Structure it with clear sections, include   │
│  citations, and provide actionable insights.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Found memories:
- (score=0.84) Research on emerging AI tools for automating competitive market analysis was conducted.
  categories: AI, market analysis, competitive intelligence, competitive analysis...
Tool search_memory executed with result: Found memories:
- (score=0.79) AlphaSense is a powerful search engine for financial and business data using AI technologies to glean insights from diverse formats.
  categories: software, AI, market a...


[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report: Evaluating the Top Five Emerging AI Tools for Automating Competitive Market Analysis   │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  This report evaluates the top five emerging AI tools designed for automating competitive market analysis. The  │
│  tools analyzed herein are AlphaSense, Similarweb, Crayon, GWI Spark, and Perplexity AI. Each tool exhibits     │
│  unique features that cater to mid-sized marketing firms aiming to enhance their competitive insights. The      │
│  report discusses these features, highlights limitations, outlines costs, and details ideal use cases. The      │
│  findings indicate that while these tools significantly enhance data-driven decision-making, the choice of      │
│  tool should align with specific business needs and capabilities. Proper integration and understanding of each  │
│  tool are essential for maximizing their analytical potential.                                                  │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│                                                                                                                 │
│  ### 1. Identification of Top AI Tools                                                                          │
│                                                                                                                 │
│  The top five emerging AI tools evaluated are:                                                                  │
│                                                                                                                 │
│  - **AlphaSense**: A comprehensive search engine suited for financial and business data.                        │
│  - **Similarweb**: Provides insights into competitors' online traffic and digital market performance.           │
│  - **Crayon**: Monitors competitive trends through real-time data tracking.                                     │
│  - **GWI Spark**: Aggregates social media data for audience insights.                                           │
│  - **Perplexity AI**: Utilizes natural language processing to answer competitive intelligence queries.          │
│                                                                                                                 │
│  ### 2. Features of AI Tools                                                                                    │
│                                                                                                                 │
│  - **AlphaSense**:                                                                                              │
│    - **Features**: Offers a powerful search capability over various business documents, employing advanced      │
│  Natural Language Processing (NLP) to extract critical insights (Forrester, 2023).                              │
│  - **Similarweb**:                                     

Report successfully saved to: research_report.md


<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**📝 我的笔记 — 这里为什么要用 `await ... kickoff_async()` 而不是 `crew.kickoff()`**

原本这个 cell 写的是 `result = deep_research_crew.kickoff(inputs=inputs)`，运行后报了：

```
RuntimeError: Agent execution was invoked synchronously from within a running
event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()`
(or `await agent.aexecute_task(...)`) when calling from async code.
```

**原因**：Jupyter 的 `ipykernel` 内核本身一直跑着一个 asyncio 事件循环；这个版本的 `crewai` 内部走的是异步执行路径，检测到"当前已经处于一个运行中的事件循环"时就拒绝同步调用。纯 `.py` 脚本没有预先存在的事件循环，所以同样代码在脚本里能跑通，只有 notebook 里会炸。和上一节课（m1 lab, `exercise.ipynb`）里 `crew.kickoff()` 报的是同一个问题。

**修复**：改成 `result = await deep_research_crew.kickoff_async(inputs=inputs)`（Jupyter 支持 cell 里直接 top-level `await`）。也可以保留同步写法，开头加一次 `nest_asyncio.apply()` 打补丁。

</div>


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧪 看结果

</div>

`result` 是 `CrewOutput`：`.raw` 是最终报告，`.tasks_output` 是每个 task 的输出，`.token_usage` 是总用量。hook 已经把报告写进 `research_report.md`。

In [12]:
from IPython.display import Markdown, display

for t in result.tasks_output:                       # 每个 task 的输出长度，看看信息量怎么流动
    print(f"{t.agent:<22} {len(t.raw):>6} chars | {t.description[:60]}...")
print(result.token_usage)
display(Markdown(Path("research_report.md").read_text()))   # hook 存下的最终报告

Research Planner         2505 chars | Break down the research query "Evaluate the top five emergin...
Internet Researcher      6317 chars | Using the research plan, search the internet and scrape rele...
Fact Checker             4707 chars | Review all collected research data for accuracy and consiste...
Report Writer            6655 chars | Create a comprehensive report that answers the original quer...
total_tokens=319962 prompt_tokens=307232 cached_prompt_tokens=109696 completion_tokens=12730 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=77


# Comprehensive Report: Evaluating the Top Five Emerging AI Tools for Automating Competitive Market Analysis

## Executive Summary

This report evaluates the top five emerging AI tools designed for automating competitive market analysis. The tools analyzed herein are AlphaSense, Similarweb, Crayon, GWI Spark, and Perplexity AI. Each tool exhibits unique features that cater to mid-sized marketing firms aiming to enhance their competitive insights. The report discusses these features, highlights limitations, outlines costs, and details ideal use cases. The findings indicate that while these tools significantly enhance data-driven decision-making, the choice of tool should align with specific business needs and capabilities. Proper integration and understanding of each tool are essential for maximizing their analytical potential.

## Detailed Findings

### 1. Identification of Top AI Tools

The top five emerging AI tools evaluated are:

- **AlphaSense**: A comprehensive search engine suited for financial and business data.
- **Similarweb**: Provides insights into competitors' online traffic and digital market performance.
- **Crayon**: Monitors competitive trends through real-time data tracking.
- **GWI Spark**: Aggregates social media data for audience insights.
- **Perplexity AI**: Utilizes natural language processing to answer competitive intelligence queries.

### 2. Features of AI Tools

- **AlphaSense**:
  - **Features**: Offers a powerful search capability over various business documents, employing advanced Natural Language Processing (NLP) to extract critical insights (Forrester, 2023).
- **Similarweb**:
  - **Features**: Delivers web traffic analytics, audience demographics, and engagement metrics to benchmark against competitors' performances (Business Yield, 2023).
- **Crayon**:
  - **Features**: Real-time monitoring of market activities with alerts based on changes in competitive dynamics (Business Yield, 2023).
- **GWI Spark**:
  - **Features**: Provides market intelligence through consumer-generated data from social media platforms, enabling targeted marketing strategies (AlphaSense, 2023).
- **Perplexity AI**:
  - **Features**: Engages users in natural language to deliver data insights intuitively and effectively (Forbes Business Council, 2023).

### 3. Limitations of AI Tools

Despite their advantages, there are notable limitations:

- **AlphaSense**: Heavily reliant on indexed data quality, potentially missing localized content, limiting its relevance for specific markets (Forrester, 2023).
- **Similarweb**: Data is often estimated, which may incur inaccuracies in quantitative analytics (Business Yield, 2023).
- **Crayon**: Primarily focused on real-time tracking; it may lack depth in analytical insights without human interpretation (Business Yield, 2023).
- **GWI Spark**: Costs and insights may vary significantly depending on the subscription level, potentially burdening smaller firms (GWI Spark, 2023).
- **Perplexity AI**: The accuracy of outputs is contingent upon the sources utilized in its algorithms, which could lead to misleading information (Forbes Business Council, 2023).

### 4. Cost Analysis

- **AlphaSense**: Pricing ranges from $10,000 to $40,000 annually, tailored for corporate strategy (Forrester, 2023).
- **Similarweb**: Starts at approximately $125/month for entry-level features, with custom pricing based on business requirements (Business Yield, 2023).
- **Crayon**: Pricing begins at around $15,000 per year, focusing on competitive insights for sales teams (Business Yield, 2023).
- **GWI Spark**: Tiered pricing starts at around $150/user/month for its consumer insight capabilities (GWI Spark, 2023).
- **Perplexity AI**: Base features are available for free, with advanced options for $20/month (Forbes Business Council, 2023).

### 5. Ideal Use Cases for Mid-Sized Marketing Firms

- **AlphaSense**: Best suited for firms requiring comprehensive insights in volatile industries for strategic decision-making.
- **Similarweb**: Ideal for digital teams benchmarked against competitive digital strategies.
- **Crayon**: Offers real-time competitive tracking beneficial for companies needing constant updates on market trends.
- **GWI Spark**: Targets businesses focused on detailed demographic insights to refine marketing strategies.
- **Perplexity AI**: Useful for rapid research and fact-checking in fast-paced environments requiring in-depth analysis.

## Supporting Evidence and Analysis

The gathered insights reflect data corroborated by credible industry sources, such as Forrester, Gartner, and specific user experiences. The balance between strengths and limitations suggests that while these AI tools substantially improve competitive analysis capabilities, the discrepancies in pricing structures may hinder accessibility for some mid-sized firms. 

The diversification in features also calls for a careful consideration of specific operational needs when selecting an AI tool for market analysis to ensure optimal integration and functionality.

## Complete Source Citations

1. Forrester. (2023). The Forrester Wave™: Market And Competitive Intelligence Platforms, Q2 2023. Retrieved from [Forrester](https://www.forrester.com/report/the-forrester-wave-tm-market-and-competitive-intelligence-platform-q2-2023/RES178487)
2. Business Yield. (2023). 11 BEST COMPETITIVE INTELLIGENCE TOOLS IN 2023. Retrieved from [Business Yield](https://businessyield.com/tech/technology/11-best-competitive-intelligence-tools/)
3. Forbes Business Council. (2023). Harnessing AI To Take Competitor Analysis To The Next Level. Retrieved from [Forbes](https://www.forbes.com/counsels/forbesbusinesscouncil/2023/10/19/harnessing-ai-to-take-competitor-analysis-to-the-next-level/)
4. GWI Spark. (2023). Market Intelligence Insights. Retrieved from [GWI](https://www.gwi.com)

## Key Insights and Recommendations

- **Recommendation for Tool Selection**: Mid-sized marketing firms should evaluate their specific analytical needs and budget constraints before choosing a tool. Aligning the tool's capabilities with the firm's strategic goals is crucial.
- **Integration Consideration**: Investing in tools that integrate seamlessly with existing analytics frameworks will enhance overall effectiveness and efficiency.
- **Regular Assessment**: Continually assessing the ROI and effectiveness of the chosen AI tool will support strategic adjustments and optimize marketing efforts.

By systematically leveraging these AI tools, marketing firms can better position themselves in a competitive landscape, making informed, data-driven decisions that drive business success.

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| 在 notebook 里用 `crew.kickoff()` | `RuntimeError: ... invoked synchronously from within a running event loop` | 改用 `await crew.kickoff_async(...)` |
| guardrail 写成两个参数 / 不返回 tuple | 用单数 `guardrail=` 时建 Task 就报 `ValueError: Guardrail function must accept exactly one parameter`；用 `guardrails=[...]` 则到运行时才出错 | 只收一个 `output`，返回 `(bool, 值或错误说明)` |
| guardrail 里直接 `output.lower()` | `AttributeError: 'TaskOutput' object has no attribute 'lower'` | 运行时收到的是 `TaskOutput`，先取 `.raw` |
| `inputs` 的 key 和 YAML 占位符不一致 | `ValueError: Missing required template variable 'user_query' in description` | YAML 写 `{user_query}`，inputs 就用 `"user_query"` |
| `memory=True` 却没有 OpenAI key | 保存/召回记忆时报 embedder 错误 | 默认 embedder 是 OpenAI；或 `Crew(embedder={...})` 换提供商 |
| 以为每次运行都是「干净」的 | 报告里出现上次题目的内容 | memory 持久化在 `~/Library/Application Support/<文件夹名>/memory`，删掉它即可重置 |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. guardrail 函数的签名和返回值有什么规定？</b></summary>

只接受一个参数（task 的输出，运行时是 `TaskOutput`）；返回 `(True, 结果)` 表示通过并把结果传下去，`(False, "说明")` 表示失败。

</details>

<details><summary><b>2. guardrail 失败后会发生什么？</b></summary>

错误说明作为反馈交给 agent 重做该 task，最多重试 `guardrail_max_retries`（默认 3）次；仍失败则抛异常，crew 中止。

</details>

<details><summary><b>3. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">after_kickoff_callbacks</code> 什么时候调用？收到什么？</b></summary>

整个 crew 所有 task 结束后调用，参数是 `CrewOutput`；`result.tasks_output[-1].raw` 是最后一个 task 的输出。

</details>

<details><summary><b>4. 为什么 tasks.yaml 里不需要写 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context</code>？</b></summary>

sequential 流程下，每个 task 自动拿到前面所有 task 的输出作为 context。

</details>

<details><summary><b>5. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">memory=True</code> 在 crewai 1.15 里做了什么？存在哪？</b></summary>

启用统一 `Memory`：task 开始前按 task 描述召回相关记忆拼进 prompt，结束后把输出提炼成记忆存入 LanceDB（OpenAI embedding）；路径是 `~/Library/Application Support/<cwd 文件夹名>/memory`，跨运行持久。

</details>

<details><summary><b>6. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">max_rpm</code> 和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">max_iter</code> 分别限制什么？</b></summary>

`max_rpm`：每分钟 LLM 请求数上限（防限流）；`max_iter`：agent 推理/调工具的最大轮数（默认 25），到上限必须给出当前最好答案。

</details>

<details><summary><b>7. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">CREWAI_TESTING=true</code> 有什么用？</b></summary>

让 crewai 视为测试环境，跳过首次运行时询问是否开启 tracing 的交互提示，避免 notebook 卡住。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**guardrail 管单个 task 的输出质量（不合格就打回重做），hook 管 crew 结束后的收尾（存文件），memory 让 agent 跨 task / 跨运行带着记忆工作**；三者都是在不改 agent 的前提下加可靠性。

</div>